In [2]:
import os
import torch
import numpy as np
from PIL import Image
import pickle

# -------------------------
# LOAD TRAJECTORIES (GT)
# -------------------------
with open("splined_trajectories_3.txt", "rb") as f:
    trajectories = pickle.load(f)

data_root = "trajectory_images"

all_images = []
all_coords = []
all_targets = []

# -------------------------
# LOOP THROUGH TRAJECTORY FOLDERS
# -------------------------
for traj_folder in sorted(os.listdir(data_root)):

    traj_path = os.path.join(data_root, traj_folder)

    if not os.path.isdir(traj_path):
        continue

    # -------------------------
    # 🔥 EXTRACT INDEX SAFELY
    # -------------------------
    try:
        traj_idx = int(traj_folder.split("_")[-1])
    except:
        print(f"Skipping folder (bad name): {traj_folder}")
        continue

    # get correct trajectory
    traj = np.array(trajectories[traj_idx])  # (T,4)

    # -------------------------
    # COMPUTE TARGET
    # -------------------------
    left = traj[:, :2]
    right = traj[:, 2:]

    if np.max(np.linalg.norm(left - left[0], axis=1)) > np.max(np.linalg.norm(right - right[0], axis=1)):
        target = left[-1]
    else:
        target = right[-1]

    # -------------------------
    # LOAD IMAGES
    # -------------------------
    frames = []
    frame_files = sorted(os.listdir(traj_path))

    for frame_file in frame_files:
        img_path = os.path.join(traj_path, frame_file)

        img = Image.open(img_path).convert("RGB")

        img_np = np.array(img)  # (H, W, 3)

        # normalize to [0,1]
        img_tensor = torch.from_numpy(img_np).float() / 255.0

        # convert to (C,H,W)
        img_tensor = img_tensor.permute(2, 0, 1)

        frames.append(img_tensor)

    # -------------------------
    # STACK FRAMES
    # -------------------------
    traj_images = torch.stack(frames)            # (T, C, H, W)
    traj_coords = torch.from_numpy(traj).float() # (T, 4)
    traj_target = torch.from_numpy(target).float()  # (2,)

    # -------------------------
    # SANITY CHECK (IMPORTANT)
    # -------------------------
    if traj_images.shape[0] != traj_coords.shape[0]:
        print(f"Mismatch in {traj_folder}, skipping...")
        continue

    # append
    all_images.append(traj_images)
    all_coords.append(traj_coords)
    all_targets.append(traj_target)

# -------------------------
# FINAL STACK
# -------------------------
all_images = torch.stack(all_images)    # (N, T, C, H, W)
all_coords = torch.stack(all_coords)    # (N, T, 4)
all_targets = torch.stack(all_targets)  # (N, 2)

# -------------------------
# SAVE
# -------------------------
torch.save({
    "images": all_images,
    "coords": all_coords,
    "targets": all_targets
}, "trajectory_dataset.pt")

# -------------------------
# PRINT SHAPES
# -------------------------
print("Images shape :", all_images.shape)
print("Coords shape :", all_coords.shape)
print("Targets shape:", all_targets.shape)

Images shape : torch.Size([498, 21, 3, 128, 128])
Coords shape : torch.Size([498, 21, 4])
Targets shape: torch.Size([498, 2])
